# 通过 API 触发 Workspace Agent

Workspace Agents 让团队可以在 ChatGPT 中保存可重复执行的工作：指令、关联的上下文、应用操作、审批和输出格式。API 触发器则让另一个系统能够在工作从 ChatGPT 之外开始时启动这个已保存的工作流。

在本 notebook 中，你将发送一个源事件，确认触发已被接受，并在智能体的目标位置中验证结果。API 负责启动运行，而指令、应用权限和审批设置决定接下来会发生什么。

*通过 API 触发的运行是异步的。该端点会将运行加入队列，但不会返回智能体完成后的响应。*


## 你将构建什么

完成后，你将拥有一个已保存的 Workspace Agent、一个带有 `agtch_...` 触发器 ID 的 API 渠道，以及一段可以针对你的智能体实际运行的 Python 触发调用。


## 前提条件

在运行实际触发的单元格之前，你需要：

| 你需要 | 参考文档 |
| --- | --- |
| 在受支持的 ChatGPT 工作区中启用 Workspace Agents，并拥有创建和共享智能体的权限 | [ChatGPT Workspace Agents 指南](https://help.openai.com/en/articles/20001143-chatgpt-workspace-agents-for-enterprise-and-business) |
| 一个已保存的 Workspace Agent，且它有一个可以写入的目标位置 | [创建和管理 Workspace Agents](https://help.openai.com/en/articles/20001143-chatgpt-workspace-agents-for-enterprise-and-business) |
| 该智能体上的一个 API 渠道及其 `agtch_...` 触发器 ID | [触发 workspace agent 运行](https://developers.openai.com/workspace-agents/trigger-runs) |
| 一个具有 Workspace Agents 权限范围的 Workspace Agent 访问令牌 | [使用 Workspace Agent 访问令牌进行认证](https://developers.openai.com/workspace-agents/authentication) |
| 一个能够发送 HTTPS `POST` 请求的 notebook 或后端环境 | 本 notebook 使用 Python 标准库，不需要 OpenAI Platform SDK。 |

有一个异步细节很重要：HTTP 请求会在智能体完成之前就返回。目标位置可以是智能体能够写入的任何已连接应用或界面，例如频道、文档、电子邮件或工单。

请查看帮助中心，了解当前的套餐和地区可用性。


## 架构

源系统决定工作何时开始。Workspace Agent 则利用其指令、上下文、应用和审批来决定已保存的工作流如何运行。

<p align="center">
  <img src="../../../images/business-technical-bridge.svg" alt="Flow from source event to backend, API trigger, Workspace Agent, and output" width="80%" />
</p>

*后端启动运行。智能体遵循已保存的工作流并写入结果。*


## 1. 编写智能体指令

API 触发器只负责启动运行。Workspace Agent 的指令告诉智能体接下来要做什么。

| 由指令决定 | 由应用设置决定 |
| --- | --- |
| 使用哪个目标位置 | 应用能否访问该目标位置 |
| 写入什么输出 | 写入操作是否可用 |
| 立即执行还是先询问 | 写入前是否需要审批 |

在本演示中，指令应当范围明确：读取源事件，向目标位置写入一条更新，并包含请求 ID 以便你进行验证。

<p align="center">
  <img src="../../../images/agent_instructions.png" alt="Workspace Agent builder showing channels, connected apps, files, and instructions" width="80%" />
</p>

*使用指令部分明确说明目标位置、写入行为和审批预期。*


In [ ]:
agent_builder_prompt = """
Create a Workspace Agent that writes API-triggered updates to the configured destination.

When the agent receives an API-triggered run:
1. Find the request ID, source system, output destination, and update text in the input.
2. Write one short update to that destination if the connected app allows it.
3. Start the update with: Workspace Agent API demo received: <Request ID>
4. Include the source system, submitted timestamp, and update text if present.
5. After the write succeeds, stop.

Use the exact destination named in the input or saved in these instructions.
Do not assume the API-triggered run has a current chat, channel, thread, or document context.
If write approval is required, request approval. If the destination is unavailable, say that the output destination could not be reached.
""".strip()

print(agent_builder_prompt)


在添加 API 触发器之前，先在 Preview 中测试确切的行为。粘贴一条包含请求 ID、目标位置和更新文本的示例消息。如果 Preview 无法写入目标位置，API 触发器也同样无法写入。


## 2. 连接一个输出目标位置

选择一个智能体可以写入的低风险目标位置：频道、文档、电子邮件、工单或其他已连接的应用。在测试 API 触发器之前，配置好应用连接，并决定写入是否需要审批。

自动批准的写入适用于低风险的演示目标位置。风险较高的写入应保持需要审批。

| 认证模式 | 适用场景 | 注意事项 |
| --- | --- | --- |
| 最终用户账户 | 每个运行智能体的人都应使用自己的应用权限 | 通过 API 触发的工作流可能要求调用方拥有相应的应用访问权限 |
| 智能体所属账户 | 工作流需要一个一致的共享连接，例如团队频道、共享文档或共享收件箱 | 尽可能使用服务账户。除非你了解风险，否则避免使用个人账户 |


In [ ]:
api_output_instructions = """
When triggered by API, write one short update to the configured destination.
Always include the request ID from the trigger input.

Start the update with: Workspace Agent API demo received: <Request ID>
Include the source system, submitted timestamp, and update text if present.
Use the exact destination named in the input or saved in these instructions.
Do not assume the API-triggered run has a current chat, channel, thread, or document context.
If writes are auto-approved for this destination, write immediately. If approval is required, request approval.
After the write succeeds, stop.
""".strip()

print(api_output_instructions)


## 3. 添加 API 渠道

在 Workspace Agent 构建器中打开智能体。添加一个 API 渠道，保存智能体，复制触发器 ID，并确认该 ID 以 `agtch_` 开头。

将智能体共享给其访问令牌将用于调用 API 的用户。触发器 ID 放在 API 路径中，它与智能体名称不同。

<p align="center">
  <img src="../../../images/api_channel.png" alt="API channel with trigger ID" width="80%" />
</p>

*在触发端点中使用这个 API 渠道 ID。*


## 4. 配置实际调用

你需要两个值：

| 值 | 来源 |
| --- | --- |
| API 触发器 ID | 智能体的 API 渠道。以 `agtch_...` 开头。 |
| Workspace Agent 访问令牌 | ChatGPT `Admin > Access tokens`，需具有 Workspace Agents 权限范围。 |

对于 `https://api.chatgpt.com`, 请使用 Workspace Agent 访问令牌，而不是 OpenAI Platform API 密钥。令牌所有者必须能够运行该共享智能体，否则 API 会返回 `403 Forbidden`。

<p align="center">
  <img src="../../../images/access_token.png" alt="Access token settings with Workspace Agents scope" width="80%" />
</p>

*将令牌作为密钥存储。不要把它粘贴到 notebook 代码中。*


要进行实际运行，请在打开 notebook 之前导出这些值：

```bash
export WORKSPACE_AGENT_TRIGGER_ID="agtch_..."
export WORKSPACE_AGENT_ACCESS_TOKEN="<workspace-agent-access-token>"
export WORKSPACE_AGENT_OUTPUT_DESTINATION="output-destination"

jupyter notebook  # or jupyter lab
```

`WORKSPACE_AGENT_OUTPUT_DESTINATION` 是可选的。它在触发输入中标明目标位置；访问权限仍然来自智能体已连接的应用。


In [ ]:
import json
import os
import textwrap
import urllib.error
import urllib.request
from datetime import datetime, timezone


def read_env(name: str, default: str = "") -> tuple[str, str]:
    value = os.environ.get(name, "").strip()
    if value:
        return value, "environment"
    return default, "default" if default else "missing"


TRIGGER_ID, trigger_id_source = read_env("WORKSPACE_AGENT_TRIGGER_ID")
ACCESS_TOKEN, access_token_source = read_env("WORKSPACE_AGENT_ACCESS_TOKEN")
OUTPUT_DESTINATION, output_destination_source = read_env(
    "WORKSPACE_AGENT_OUTPUT_DESTINATION",
    "the agent's configured destination",
)
API_BASE = "https://api.chatgpt.com/v1"

print(f"Trigger ID configured: {bool(TRIGGER_ID)} ({trigger_id_source})")
print(f"Access token configured: {bool(ACCESS_TOKEN)} ({access_token_source})")
print(f"Output destination label: {OUTPUT_DESTINATION} ({output_destination_source})")

if ACCESS_TOKEN.startswith("sk-"):
    print("Warning: this looks like an OpenAI Platform API key. Create a ChatGPT Workspace Agent access token from Admin > Access tokens instead.")

if OUTPUT_DESTINATION == "the agent's configured destination":
    print("Tip: set WORKSPACE_AGENT_OUTPUT_DESTINATION when you want the API input to name a specific destination.")


## 5. 发送一个源事件

让第一个输入保持简短：源系统发送一个事件，Workspace Agent 向其目标位置写入一条更新。


In [ ]:
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

source_event = {
    "request_id": f"DEMO-{run_stamp}",
    "submitted_at": datetime.now(timezone.utc).isoformat(),
    "source_system": "notebook demo",
    "update_text": "The API trigger reached the Workspace Agent and asked it to write this update.",
    "output_destination": OUTPUT_DESTINATION,
}

source_event


在调用触发端点之前，先把源事件映射为纯文本。API 将事件传递给智能体，而智能体已保存的指令、应用权限和审批设置控制写入行为。


In [ ]:
def build_agent_input(event: dict) -> str:
    return textwrap.dedent(f"""
    Request ID: {event['request_id']}
    Source system: {event['source_system']}
    Submitted at: {event['submitted_at']}
    Output destination: {event['output_destination']}
    Update text: {event['update_text']}

    Please write this update to the configured destination.
    Start with exactly: Workspace Agent API demo received: {event['request_id']}
    """).strip()


agent_input = build_agent_input(source_event)
print(agent_input)


当多个事件应延续同一个智能体对话时（例如针对同一请求的多次更新），请使用 `conversation_key`。

重试时，请发送 `Idempotency-Key` 请求头。仅在重试同一个源事件时才复用同一个键。


In [ ]:
conversation_key = f"workspace-agent-api-demo-{source_event['request_id']}"
idempotency_key = f"workspace-agent-api-demo-{source_event['request_id']}-v1"

payload = {
    "conversation_key": conversation_key,
    "input": agent_input,
}

print(json.dumps(payload, indent=2))
print("Idempotency-Key:", idempotency_key)


## 6. 通过 API 触发智能体

端点：

```http
POST https://api.chatgpt.com/v1/workspace_agents/{id}/trigger
```

必需和推荐的请求字段：

| 位置 | 字段 | 是否必需 | 说明 |
| --- | --- | --- | --- |
| Path | `id` | 是 | 来自智能体 API 渠道的 API 触发器 ID，格式为 `agtch_...` |
| Header | `Authorization` | 是 | 使用 Workspace Agent 访问令牌的 Bearer 凭据 |
| Header | `Content-Type` | 是 | 使用 `application/json` |
| Header | `Idempotency-Key` | 推荐 | 用于重试同一源事件的稳定键 |
| Body | `input` | 是 | 作为触发输入传递给智能体的消息文本 |
| Body | `conversation_key` | 否 | 由调用方定义的稳定 ID，用于延续同一个智能体对话 |


In [ ]:
def trigger_workspace_agent(trigger_id: str, access_token: str, payload: dict, idempotency_key: str) -> dict:
    url = f"{API_BASE}/workspace_agents/{trigger_id}/trigger"
    body = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        url,
        data=body,
        method="POST",
        headers={
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json",
            "Idempotency-Key": idempotency_key,
        },
    )

    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            return {"ok": True, "status": response.status, "error": None}
    except urllib.error.HTTPError as exc:
        error_body = exc.read().decode("utf-8", errors="replace")
        try:
            parsed_error = json.loads(error_body).get("error", {})
        except json.JSONDecodeError:
            parsed_error = {"message": error_body}
        return {"ok": False, "status": exc.code, "error": parsed_error}


def print_trigger_error(result: dict) -> None:
    status = result.get("status")
    error = result.get("error") or {}
    message = error.get("message", "Unknown error")
    code = error.get("code")

    print(f"Workspace Agent trigger failed: HTTP {status}")
    print("Message:", message)
    if code:
        print("Code:", code)

    if status == 401:
        print()
        print("Fix: create a fresh ChatGPT Workspace Agent access token from Admin > Access tokens with the Workspace Agents scope.")
        print("Do not use an OpenAI Platform API key from platform.openai.com for this endpoint.")
    elif status == 403:
        print()
        print("Fix: the token is valid, but the token owner cannot trigger this agent. Share the agent with that user or use a token from an allowed owner.")
    elif status == 404:
        print()
        print("Fix: confirm the API channel trigger ID starts with agtch_ and belongs to an agent visible to the token owner.")
    elif status == 409:
        print()
        print("Fix: save/publish the agent and confirm the API channel is active and runnable.")


if not TRIGGER_ID or not ACCESS_TOKEN:
    print("No live request sent.")
    print("Set WORKSPACE_AGENT_TRIGGER_ID and WORKSPACE_AGENT_ACCESS_TOKEN before starting the kernel.")
else:
    result = trigger_workspace_agent(TRIGGER_ID, ACCESS_TOKEN, payload, idempotency_key)
    if not result["ok"]:
        print_trigger_error(result)
    else:
        status = result["status"]
        print("HTTP status:", status)
        assert status == 202, f"Expected 202 Accepted, got {status}"
        print("Trigger accepted. Look in the configured destination for the result.")
        print("Search for request ID:", source_event["request_id"])
        print("Expected output destination:", source_event["output_destination"])


`202 Accepted` 表示触发已被接受并等待处理，并不意味着智能体已经完成。API 不返回响应体、不返回公开的运行 ID，也不返回智能体的最终输出。

第一次实际运行时，请留意三件事：`HTTP status: 202`、一个唯一的请求 ID，以及目标位置中一条以 `Workspace Agent API demo received:` 开头的更新。

如果触发成功但没有写入任何内容，请排查智能体配置，而不是 HTTP 请求。通常是以下某一项缺失：目标位置指令、应用权限、应用认证或预期的审批设置。

在 notebook 中反复测试时，请确保生成的请求 ID 和幂等键保持唯一。对同一个源事件复用相同的 `Idempotency-Key`，可能会返回最初被接受的结果，而不是将新的运行加入队列。


## 7. 将调用改造为后端 worker

上面的实际调用是最小测试。下面的辅助函数将相同的请求结构封装起来，供后端 worker、定时任务或工作流引擎使用。

对于同一个源事件，请保持 `conversation_key` 和 `Idempotency-Key` 稳定，以免重试时将重复的运行加入队列。


In [ ]:
def build_backend_trigger_request(event: dict) -> dict:
    """Create the Workspace Agent trigger request from a source-system event."""
    request_id = event["request_id"]
    return {
        "trigger_id": TRIGGER_ID,
        "payload": {
            "conversation_key": f"workspace-agent-api-demo-{request_id}",
            "input": build_agent_input(event),
        },
        "idempotency_key": f"workspace-agent-api-demo-{request_id}-v1",
    }


def handle_source_event(event: dict, *, live: bool = False) -> dict:
    """Preview or send the backend trigger call."""
    trigger_request = build_backend_trigger_request(event)
    result = {
        "request_id": event["request_id"],
        "conversation_key": trigger_request["payload"]["conversation_key"],
        "idempotency_key": trigger_request["idempotency_key"],
        "queued": False,
        "status": None,
    }

    if live:
        if not TRIGGER_ID or not ACCESS_TOKEN:
            raise RuntimeError("Set WORKSPACE_AGENT_TRIGGER_ID and WORKSPACE_AGENT_ACCESS_TOKEN before live=True.")
        trigger_result = trigger_workspace_agent(
            trigger_request["trigger_id"],
            ACCESS_TOKEN,
            trigger_request["payload"],
            trigger_request["idempotency_key"],
        )
        result["status"] = trigger_result["status"]
        result["queued"] = trigger_result["ok"] and trigger_result["status"] == 202
        if not trigger_result["ok"]:
            result["error"] = trigger_result["error"]

    return result


backend_preview = handle_source_event(source_event, live=False)
print(json.dumps(backend_preview, indent=2))


在生产环境中，请记录源请求 ID、触发器 ID、幂等键、时间戳和 HTTP 状态码。不要记录访问令牌或敏感的请求详情。


## 8. 验证目标位置中的操作

API 不会返回完成后的响应，因此请在智能体的目标位置中验证操作。检查触发输入中指定的、或智能体指令中保存的确切目标位置。

<p align="center">
  <img src="../../../images/api-trigger-check.svg" alt="API trigger flow showing accepted request and destination check" width="80%" />
</p>

*HTTP 响应告诉你触发已被接收。请在目标位置中验证实际完成的工作。*


在配置的目标位置中搜索实际触发单元格打印出的生成请求 ID。该更新应以如下内容开头：

```text
Workspace Agent API demo received: DEMO-20260618010101
```

到这一步，端到端路径就已经打通：触发器、令牌、共享、指令、应用权限和审批行为。

如果触发成功但没有出现输出，请先检查以下几项：

| 检查项 | 需要验证的内容 |
| --- | --- |
| Preview | 粘贴确切的 `agent_input` 文本，确认智能体会写入目标位置 |
| 目标位置 | 在启动内核之前，将 `WORKSPACE_AGENT_OUTPUT_DESTINATION` 设置为确切的目标位置名称 |
| 权限 | 确认已连接的应用在 API 触发的运行中可以写入该目标位置 |
| 审批 | 如果演示应自动写入，请确认写入操作已设为自动批准 |
| 访问 | 确认令牌所有者可以运行该智能体，且已保存的 API 渠道处于活动状态 |

在此基础上，你可以用任何后端事件替换 `source_event`。API 调用保持不变，由智能体所有者在 ChatGPT 中更新指令、目标位置和审批行为。


在推广之前，先运行一个小型测试矩阵：

| 测试 | 预期结果 |
| --- | --- |
| 正常路径 | 智能体将 `Workspace Agent API demo received: <request_id>` 写入配置的目标位置 |
| 缺少请求 ID | 智能体询问请求 ID，且不写入含糊的更新 |
| 使用相同幂等键的重复重试 | API 返回最初被接受的结果，而不是将第二个触发事件加入队列 |
| 来自未共享用户的令牌 | API 返回 `403 Forbidden` |
| 错误的触发器 ID | API 返回 `404 Not Found` |
| 智能体或目标位置无法运行 | API 返回 `409 Conflict` |


## 错误处理

触发端点可能返回：

| 状态码 | 含义 | 处理方法 |
| --- | --- | --- |
| `202 Accepted` | 请求已被接受并等待处理 | 在配置的目标位置中验证结果 |
| `401 Unauthorized` | 令牌缺失、已过期、已撤销、无效，或凭据类型错误 | 在 `Admin > Access tokens` 中创建具有 Workspace Agents 权限范围的 ChatGPT Workspace Agent 访问令牌。不要使用 OpenAI Platform API 密钥 |
| `403 Forbidden` | 令牌有效，但无法触发该智能体 | 将智能体共享给调用方，或使用来自被允许用户的令牌 |
| `404 Not Found` | 触发器 ID 不存在，或对调用方不可见 | 确认来自 API 渠道的 `agtch_` ID |
| `409 Conflict` | 智能体或 API 渠道无法运行 | 保存/发布智能体，并确认 API 渠道处于活动状态 |
| `202 Accepted`，但没有出现输出 | 触发成功，但智能体没有写入目标位置 | 检查指令、应用认证、目标位置权限和写入审批设置 |

对于暂时性的网络故障，请针对同一个源事件使用相同的 `Idempotency-Key` 重试。对于不同的源事件，请生成新的幂等键。


## 生产环境检查清单

在试点之外扩大使用范围之前，请确定以下四个方面：

| 方面 | 检查项 |
| --- | --- |
| 认证 | 妥善存储并轮换访问令牌。选择能清楚体现运行归属的令牌所有者。 |
| 访问 | 仅将智能体共享给被允许的用户或群组。确认已连接的应用可以写入目标位置。 |
| 行为 | 在智能体指令中明确目标位置、输出格式和审批行为。 |
| 运维 | 包含源事件 ID，监控非 202 响应，并留意缺失的输出。 |

只对低风险写入启用自动批准。在与业务负责人审查工作流之前，敏感写入应保持需要审批。


## 何时使用这种模式

当应由业务部门拥有的 Workspace Agent 来完成工作、但由另一个系统启动它时，请使用 API 触发器。

| 在以下情况使用这种模式 | 在以下情况使用其他方法 |
| --- | --- |
| 应由非技术负责人维护工作流指令。 | 调用方需要在 HTTP 响应中获得完成后的答案。 |
| 工作由表单、工单、CRM 记录、定时任务或后端事件发起。 | 源系统需要低延迟的同步行为。 |
| 结果可以写入已连接的应用、频道、文档或电子邮件。 | 智能体输出没有持久的目标位置。 |
| 指令和审批设置可以安全地控制写入行为。 | 工作流影响重大，且缺乏审查、审计或回滚途径。 |

这就是职责划分：业务团队在 ChatGPT 中完善智能体，而工程团队负责保持源系统触发器的稳定。


## 改造这种模式

当工作从 ChatGPT 之外开始、但应通过可复用的 Workspace Agent 运行时，可以使用同样的模式：

| 工作流 | 源事件 | 智能体输出 |
| --- | --- | --- |
| 会议跟进 | 会议录音或转录文本已就绪 | 行动项和跟进草稿 |
| 日历驱动的准备工作 | 发现 Google Calendar 事件或即将召开的会议日程 | 简报文档或准备摘要 |
| 入站潜在客户资格评估 | CRM 潜在客户或表单提交 | 客户摘要和销售任务 |
| 支持升级 | 创建了高优先级工单 | 升级摘要和后续步骤 |
| 员工支持 | 创建了服务台工单 | 分诊答复或升级路径 |
| 每周报告 | 定时任务发现新记录 | 摘要文档或团队更新 |

从一个范围明确的工作流、一个清晰的源事件和一个输出目标位置开始。在智能体的行为保持稳定之后，再添加更多目标位置、更丰富的上下文或更多需要审批的操作。


## 延伸阅读

- [在 ChatGPT 中构建 workspace agents，完成可重复的端到端工作](https://developers.openai.com/cookbook/articles/chatgpt-agents-sales-meeting-prep)
- [ChatGPT Workspace Agents for Enterprise and Business](https://help.openai.com/en/articles/20001143-chatgpt-workspace-agents-for-enterprise-and-business)
- [触发 workspace agent 运行](https://developers.openai.com/workspace-agents/trigger-runs)
- [使用 Workspace Agent 访问令牌进行认证](https://developers.openai.com/workspace-agents/authentication)